In [25]:
import sys
import time
from pathlib import Path
root = Path("~/Desktop/heuristics/heuristics/src").expanduser()
sys.path.append(str(root.absolute()))
import numpy as np
from structures import parse_instance, objective, Solution
from local_search import local_search, StepFunction, MaxIterations
from tabu import tabu_search   # adjust import paths
from construction import construction
from neighborhoods import IntraRouteNeighborhood, PairRelocateNeighborhood, TwoOptNeighborhood
from sa import simulated_annealing


In [22]:
# INSTANCE_PATH = "../instances/100/train/instance1_nreq100_nveh2_gamma89.txt"
# INSTANCE_PATH = "../instances/50/train/instance1_nreq50_nveh2_gamma50.txt"
# INSTANCE_PATH = "../instances/10000/train/instance1_nreq10000_nveh200_gamma8674.txt"

# INSTANCE_PATH = "/home/chris/Desktop/heuristics/heuristics/instances/100/train/instance1_nreq100_nveh2_gamma89.txt"
# INSTANCE_PATH = "/home/chris/Desktop/heuristics/heuristics/instances/500/train/instance1_nreq500_nveh10_gamma432.txt"
INSTANCE_PATH = "/home/chris/Desktop/heuristics/heuristics/instances/1000/train/instance1_nreq1000_nveh20_gamma890.txt"
TABU_TENURE = 20
MAX_ITERS = 200                                # increase to test scaling
NEIGHBORHOODS = [
    IntraRouteNeighborhood,
    PairRelocateNeighborhood,
    TwoOptNeighborhood,
]
STEP_FUNCTION = StepFunction.best_improvement  # or first_improvement
TABU_ATTR = lambda mov: mov.data               # simplest attribute

In [26]:
path = Path(
    "/home/chris/Desktop/heuristics/heuristics/instances/1000/train/instance1_nreq1000_nveh20_gamma890.txt"
)

# ----------------------------------------
# t0 → t1 : parsing
# ----------------------------------------
t0 = time.perf_counter()
I = parse_instance(path)
t1 = time.perf_counter()

# ----------------------------------------
# t1 → t2 : construction
# ----------------------------------------
sol0 = construction(I, 0.8)
f_sol = objective(I, sol0)
t2 = time.perf_counter()

# ----------------------------------------
# neighborhood list (Python equivalent)
# ----------------------------------------
neighborhoods = [
    IntraRouteNeighborhood,
    PairRelocateNeighborhood,
    TwoOptNeighborhood,
]

stopping = MaxIterations(500)

# ----------------------------------------
# t2 → t3 : local search
# ----------------------------------------
sol1 = local_search(
    I,
    sol0,
    neighborhoods_cls=neighborhoods,
    step_function=StepFunction.first_improvement,
    stopping_criterion=stopping,
)
f_sol1 = objective(I, sol1)
t3 = time.perf_counter()

# ----------------------------------------
# Report
# ----------------------------------------
print(f"Parsing time (ms):       {(t1 - t0) * 1e3:.3f}")
print(f"Construction time (ms):  {(t2 - t1) * 1e3:.3f}")
print(f"Local search time (ms):  {(t3 - t2) * 1e3:.3f}")
print(f"Objective construction:  {f_sol}")
print(f"Objective local search:  {f_sol1}")

Parsing time (ms):       2994.143
Construction time (ms):  1.412
Local search time (ms):  11098.074
Objective construction:  284105.5033986315
Objective local search:  269826.4570906872


In [ ]:
start = time.perf_counter()
I = parse_instance(Path(INSTANCE_PATH))
middle = time.perf_counter()
sol0 = construction(I, 0.8)


sol

end = time.perf_counter()
print(1e3*(end-middle))
print(1e3*(middle-start))

1.094516999728512
2970.0943759999063


In [24]:
objective(I, sol0)

np.float64(284105.5033986315)

In [11]:
def main():
    print("Loading instance...")
    t = []
    I = parse_instance(Path(INSTANCE_PATH))

    # Simple deterministic construction: empty routes (just to test performance)
    sol0 = Solution(routes=[[] for _ in range(I.nK)])

    # print("Running tabu search...")
    t.append( time.perf_counter())
    sol = construction(I = I, a=0.5)
    t.append( time.perf_counter())
    # sol_tabu = tabu_search(
    #     I=I,
    #     sol=sol,
    #     neighborhoods_cls=NEIGHBORHOODS,
    #     tabu_tenure=TABU_TENURE,
    #     max_iters=MAX_ITERS,
    #     aspiration_factor=0.99,
    #     tabu_attribute_func=TABU_ATTR,
    # )
    
    sol_sa = simulated_annealing(
        I=I,
        sol=sol,
        neighborhoods_cls=[IntraRouteNeighborhood, PairRelocateNeighborhood, TwoOptNeighborhood],
        T_start=1.0,
        T_end=1e-3,
        cooling=0.995,
        max_iters=20,
    )
    t.append( time.perf_counter())


    # print(f"Runtime: {t1 - t0:.4f} seconds")
    print(f"Objective: {objective(I, sol):.2f}")
    print(f"Objective: {objective(I, sol_sa):.2f}")
    
    print(np.diff(t))

In [12]:
main()

Loading instance...
Objective: 4335.09
Objective: 4249.03
[3.02946999e-04 5.86197568e+00]
